# Aula 8 — Aprendizado Não Supervisionado
## Agrupamento K-means e redução de dimensionalidade com PCA

Neste laboratório, identificaremos perfis de uso de um sistema sem receber categorias prontas. Execute as células na ordem usando **Shift + Enter**.

## 1. Importar as bibliotecas

O Google Colab já oferece as bibliotecas usadas nesta prática.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## 2. Criar dados didáticos

Os dados representam 360 usuários. As três concentrações foram criadas apenas para permitir que a turma observe o comportamento do algoritmo. A tabela final não contém uma coluna com o perfil correto.

In [ ]:
rng = np.random.default_rng(42)

grupos = [
    (120, 2.5, 12, 3),
    (120, 6.5, 27, 7),
    (120, 11.0, 48, 13)
]

partes = []
for n, sessoes, duracao, funcoes in grupos:
    parte = pd.DataFrame({
        "sessoes_semana": rng.normal(sessoes, 1.0, n),
        "duracao_media_min": rng.normal(duracao, 4.0, n),
        "funcionalidades_mes": rng.normal(funcoes, 1.4, n)
    })
    partes.append(parte)

dados = pd.concat(partes, ignore_index=True).clip(lower=0)
dados.head()

## 3. Inspecionar os dados

Antes de qualquer modelo, verifique as unidades, as faixas, os valores ausentes e o significado das colunas.

In [ ]:
dados.info()
display(dados.describe().round(2))
print("Valores ausentes por coluna:")
display(dados.isna().sum())

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(
    dados["sessoes_semana"],
    dados["duracao_media_min"],
    alpha=0.6
)
plt.xlabel("Sessões por semana")
plt.ylabel("Duração média (min)")
plt.title("Dados antes do agrupamento")
plt.grid(alpha=0.2)
plt.show()

## 4. Padronizar as características

O K-means usa distâncias. A padronização evita que uma coluna domine o cálculo apenas por possuir números maiores.

In [ ]:
colunas = [
    "sessoes_semana",
    "duracao_media_min",
    "funcionalidades_mes"
]

escalador = StandardScaler()
X_padronizado = escalador.fit_transform(dados[colunas])

pd.DataFrame(X_padronizado, columns=colunas).describe().round(2)

## 5. Comparar valores de k

A inércia sempre diminui quando o número de grupos aumenta. A silhueta compara a compactação interna e a separação entre grupos.

In [ ]:
resultados_k = []

for k in range(2, 8):
    modelo_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )
    rotulos_k = modelo_k.fit_predict(X_padronizado)
    resultados_k.append({
        "k": k,
        "inercia": modelo_k.inertia_,
        "silhueta": silhouette_score(X_padronizado, rotulos_k)
    })

comparacao_k = pd.DataFrame(resultados_k)
display(comparacao_k.round(3))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

eixos[0].plot(comparacao_k["k"], comparacao_k["inercia"], marker="o")
eixos[0].set_xlabel("Número de clusters (k)")
eixos[0].set_ylabel("Inércia")
eixos[0].set_title("Método do cotovelo")

eixos[1].plot(comparacao_k["k"], comparacao_k["silhueta"], marker="o")
eixos[1].set_xlabel("Número de clusters (k)")
eixos[1].set_ylabel("Silhueta")
eixos[1].set_title("Comparação da silhueta")

plt.tight_layout()
plt.show()

## 6. Treinar o K-means com três grupos

A escolha de `k = 3` será comparada com as outras soluções no laboratório.

In [ ]:
modelo = KMeans(
    n_clusters=3,
    random_state=42,
    n_init="auto"
)

dados["cluster"] = modelo.fit_predict(X_padronizado)
dados["cluster"].value_counts().sort_index()

## 7. Interpretar os centroides

Os centroides foram aprendidos no espaço padronizado. A transformação inversa devolve as unidades originais.

In [ ]:
centroides_originais = escalador.inverse_transform(
    modelo.cluster_centers_
)

centroides = pd.DataFrame(
    centroides_originais,
    columns=colunas
)
centroides.index.name = "cluster"
display(centroides.round(2))

In [ ]:
perfil = dados.groupby("cluster")[colunas].mean()
perfil["quantidade"] = dados.groupby("cluster").size()
display(perfil.round(2))

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    dados["sessoes_semana"],
    dados["duracao_media_min"],
    c=dados["cluster"],
    cmap="viridis",
    alpha=0.65
)
plt.xlabel("Sessões por semana")
plt.ylabel("Duração média (min)")
plt.title("Agrupamento dos perfis de uso")
plt.grid(alpha=0.2)
plt.show()

## 8. Associar um novo usuário

O novo registro deve passar pelo mesmo escalador usado no treinamento.

In [ ]:
novo = pd.DataFrame({
    "sessoes_semana": [8],
    "duracao_media_min": [32],
    "funcionalidades_mes": [9]
})

novo_padronizado = escalador.transform(novo)
cluster_novo = modelo.predict(novo_padronizado)[0]
print("Cluster do novo usuário:", cluster_novo)

## 9. Representar os dados com PCA

O PCA cria duas novas coordenadas que preservam parte da variação presente nas três características padronizadas.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_padronizado)

print("Variância por componente:", pca.explained_variance_ratio_.round(3))
print("Variância total em 2D:", pca.explained_variance_ratio_.sum().round(3))

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=dados["cluster"],
    cmap="viridis",
    alpha=0.7
)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Clusters representados com PCA")
plt.grid(alpha=0.2)
plt.show()

## 10. Laboratório: comparar k = 2, 3 e 4

A célula abaixo reúne métricas, tamanhos e centroides. Use os resultados para escolher uma solução.

In [ ]:
resumo_laboratorio = []
modelos_laboratorio = {}

for k in [2, 3, 4]:
    modelo_teste = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )
    rotulos = modelo_teste.fit_predict(X_padronizado)
    tamanhos = pd.Series(rotulos).value_counts().sort_index().tolist()
    resumo_laboratorio.append({
        "k": k,
        "inercia": modelo_teste.inertia_,
        "silhueta": silhouette_score(X_padronizado, rotulos),
        "tamanhos": tamanhos
    })
    modelos_laboratorio[k] = (modelo_teste, rotulos)

display(pd.DataFrame(resumo_laboratorio).round(3))

In [ ]:
for k, (modelo_teste, rotulos) in modelos_laboratorio.items():
    centros = pd.DataFrame(
        escalador.inverse_transform(modelo_teste.cluster_centers_),
        columns=colunas
    )
    print(f"\nCentroides para k = {k}")
    display(centros.round(2))

## 11. Experimento com um valor extremo

Antes de executar, formule uma hipótese: como um usuário muito distante pode alterar a padronização, os centroides e a silhueta?

In [ ]:
usuario_extremo = pd.DataFrame({
    "sessoes_semana": [35],
    "duracao_media_min": [140],
    "funcionalidades_mes": [28]
})

dados_extremos = pd.concat(
    [dados[colunas], usuario_extremo],
    ignore_index=True
)

escalador_extremo = StandardScaler()
X_extremo = escalador_extremo.fit_transform(dados_extremos)

modelo_extremo = KMeans(n_clusters=3, random_state=42, n_init="auto")
rotulos_extremos = modelo_extremo.fit_predict(X_extremo)

print("Silhueta antes:", round(silhouette_score(X_padronizado, dados["cluster"]), 3))
print("Silhueta depois:", round(silhouette_score(X_extremo, rotulos_extremos), 3))

centros_extremos = pd.DataFrame(
    escalador_extremo.inverse_transform(modelo_extremo.cluster_centers_),
    columns=colunas
)
display(centros_extremos.round(2))

## 12. Síntese para entregar

Responda em até cinco frases:

1. Qual valor de `k` você escolheu?
2. Qual métrica sustenta a escolha?
3. Como os grupos podem ser descritos pelos centroides?
4. Quanta variação os dois componentes do PCA preservaram?
5. O que mudou após adicionar o valor extremo?

> Uma boa conclusão combina medida quantitativa, interpretação do contexto e reconhecimento das limitações.